# MPC Sweep — Shard Runner (CPU-only)

Runs a subset of the 35 MPC cells (7 presets x 5 seeds).
Open 5 copies of this notebook in parallel, each with a different SHARD_ID.

**IMPORTANT**: Runtime -> CPU (not GPU!). MPC's mpc_planner has no device
propagation; production code is CPU-only. Decision Gate confirmed Colab T4
is 0.57x SLOWER than local 12-thread CPU for MPC.

**Resilience design** (after browser/idle disconnect bug observed 2026-05-15):
- Results + state written DIRECTLY to Drive
- Heartbeat output every 60s in sweep.runner -- silent _generate_data
  (50000 env.step calls per cell, 1-2h on 2600+ blocks) no longer kills the cell
- Stdout via `python -u` for line-buffered output
- Tail of training log mirrored to Drive every cell completion

In [ ]:
# === CONFIG: change SHARD_ID for each parallel session ===
SHARD_ID = 0       # 0, 1, 2, 3, or 4
TOTAL_SHARDS = 5
LIMIT_TIME_S = 72000  # 20h safety margin within 24h Pro+ window
print(f'Shard {SHARD_ID}/{TOTAL_SHARDS}, limit={LIMIT_TIME_S}s')

In [ ]:
# Cell 1: Mount Drive + unzip bundle
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/arcgis-farmland-mpc'
BUNDLE = f'{DRIVE_ROOT}/mpc_sweep_bundle.zip'
assert os.path.exists(BUNDLE), f'Upload mpc_sweep_bundle.zip to {DRIVE_ROOT}'
!unzip -q -o $BUNDLE -d /content
!ls /content/src

In [ ]:
# Cell 2: Install deps (no GPU libs)
!pip install -q torch geopandas libpysal opensimplex rasterio pyyaml shapely

In [ ]:
# Cell 3: Env vars + thread tuning
import os
os.environ['TEST_SRC_ROOT'] = '/content/src/test'
os.environ['ADK_SRC_ROOT']  = '/content/src/adk'
os.environ['PYTHONUNBUFFERED'] = '1'
import torch
torch.set_num_threads(os.cpu_count() or 4)
print(f'torch threads: {torch.get_num_threads()}')
print(f'cpu count: {os.cpu_count()}')

In [ ]:
# Cell 4: Generate all 35 datasets (shared across shards, ~30 min)
%cd /content/src/benchmark
import pathlib
presets = ['bishan_clone','neijiang_clone','plain_small_cons','plain_large_cons',
           'plain_medium_frag','mixed_medium_frag','hilly_small_cons']
for p in presets:
    for s in range(5):
        out = f'data_dev/{p}_seed{s}'
        if pathlib.Path(f'{out}/manifest.json').exists():
            print(f'skip {out}')
            continue
        !python -u -m generator.generate --preset presets/{p}.yaml --seed {s} --out {out}
print('Done: datasets generated')

In [ ]:
# Cell 5: Run MPC sweep for this shard. Results + state go DIRECTLY to Drive.
import pathlib
RUN_DIR = f'{DRIVE_ROOT}/sweep_run'
RESULTS_ROOT = f'{RUN_DIR}/results'
STATE_PATH   = f'{RUN_DIR}/sweep_state_mpc_shard{SHARD_ID}.json'
LOG_PATH     = f'{RUN_DIR}/sweep_mpc_shard{SHARD_ID}.log'
pathlib.Path(RUN_DIR).mkdir(parents=True, exist_ok=True)
pathlib.Path(RESULTS_ROOT).mkdir(parents=True, exist_ok=True)
%cd /content/src/benchmark
!python -u -m sweep.runner \
  --manifest sweep_manifest.csv \
  --state "$STATE_PATH" \
  --data-root data_dev \
  --results-root "$RESULTS_ROOT" \
  --only-method MPC \
  --shard {SHARD_ID}/{TOTAL_SHARDS} \
  --resume \
  --limit-time {LIMIT_TIME_S} 2>&1 | tee -a "$LOG_PATH"

In [ ]:
# Cell 6: Sanity report
import json, pathlib
from collections import Counter
state = json.loads(pathlib.Path(STATE_PATH).read_text())
mpc = [c for c in state['cells'] if c['method'] == 'MPC']
shard_cells = [c for i, c in enumerate(mpc) if i % TOTAL_SHARDS == SHARD_ID]
print(f'Shard {SHARD_ID}: {Counter(c["status"] for c in shard_cells)}')
for c in shard_cells:
    if c['status'] == 'done':
        print(f"  OK    {c['cell_id']}")
    elif c['status'] == 'failed':
        print(f"  FAIL  {c['cell_id']}: {(c.get('error') or '')[:120]}")
print(f'\nResults dir: {RESULTS_ROOT}')
print(f'State:       {STATE_PATH}')
print(f'Log:         {LOG_PATH}')